# Frame table review

Reads a built frame table (`scripts/build_frame_table.py`), its profile and report
(`scripts/profile_frame_table.py`) and a few raw shots with `fusion_eq.io.load_shot`.
Cells are run manually, top to bottom. Sections follow `docs/FRAME_TABLE.md`, "Review
notebook". Plots that need individual frames use a fixed-seed sample of shots; summary
statistics use the whole table. The first cell selects the build to review: the smoke
output by default, the full table once it exists.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fusion_eq import frame_table as ft
from fusion_eq.io import load_shot
from fusion_eq.machines import machine_config

# Build to review. Switch both to the full table once it is built:
#   TABLE_DIR = Path("../../data/processed/frame_table"); REPORT_DIR = Path("../reports/frame_table")
TABLE_DIR = Path("../../data/processed/frame_table_smoke")
REPORT_DIR = Path("../reports/frame_table_smoke")
QA_TABLE = Path("../reports/qa/qa_per_shot.parquet")
SMOKE_FILE = Path("../reports/frame_table/smoke_shots.json")
SEED = 0

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
SPLITS = [s for s in ("diii_d_train", "diii_d_public_test", "mast_public_test")
          if ft.table_path(s, TABLE_DIR).is_file()]
MACHINE_OF = {"diii_d_train": "DIII-D", "diii_d_public_test": "DIII-D", "mast_public_test": "MAST"}


def read(split, columns=None, shots=None):
    """Rows of one split of the reviewed build."""
    return ft.read_frame_table(split, columns, shots, root=TABLE_DIR)


build = json.loads((TABLE_DIR / "build.json").read_text())
index = pd.read_parquet(TABLE_DIR / "shot_index.parquet")
PARAMS = build["params"]
print(f"build {build['timestamp']}, commit {str(build['repo_commit'])[:10]}, {build['runtime_s']:.0f} s")
print("params:", PARAMS)
print("splits present:", SPLITS)

## 1. Build summary

Shot and frame counts per split, shots that produced no rows, and the acceptance table
from the report. The report is written by `scripts/profile_frame_table.py`; when it is
missing, the cell says so.

In [ ]:
summary = index.groupby(["split", "machine"], observed=True).agg(
    shots=("shot", "size"), frames=("n_frames", "sum"), errors=("error", lambda s: int(s.notna().sum())),
    nan_currents=("mag_has_nan", "sum"), no_core_thomson=("ts_core_any_valid", lambda s: int((~s.fillna(False)).sum())))
display(summary)
errors = index.loc[index["error"].notna(), ["split", "shot", "error"]]
print(f"{len(errors)} shots with errors")
display(errors)

report = REPORT_DIR / "frame_table_report.md"
if report.is_file():
    lines = report.read_text().splitlines()
    print("\n".join(l for l in lines if l.startswith("| T") or l.startswith("| Id") or l.startswith("| ---")))
else:
    print(f"no report at {report}; run scripts/profile_frame_table.py --table-dir {TABLE_DIR} --out {REPORT_DIR}")

In [ ]:
display(index[["split", "shot", "n_frames", "mag_rate_hz", "mag_has_nan", "ip_direction", "tf_direction",
               "ip_fix", "ip_shift_ms", "ts_core_any_valid", "ts_edge_any_valid", "inputs_complete_frac"]]
        .sort_values(["split", "shot"]).head(40))
print("mag_rate_hz by machine:")
display(pd.crosstab(index["machine"], index["mag_rate_hz"]))

## 2. Resampling of currents (A1, A2)

One shot of each kind in the smoke set: the raw plasma current and solenoid current
inside the EFIT window with the two frame values, `__nearest` and `__mean`, drawn over
them. Then the T8 distributions: `__nearest` minus `__mean` relative to the spread of
`__mean`, per current column and recording rate. The two forms are compared on the
ridge score in the baseline phase; this section only checks that both read the trace
correctly and shows where they differ.

In [ ]:
# Kinds of the smoke set, from the QA table, restricted to shots present in this build.
sys.path.insert(0, "../scripts")
from select_smoke_shots import select  # noqa: E402

qa = pd.read_parquet(QA_TABLE)
kinds = {kind: [s for s in shots if s in set(index["shot"])] for kind, shots in select(qa, per_kind=50).items()}
kinds = {kind: shots for kind, shots in kinds.items() if shots}
for kind, shots in kinds.items():
    print(f"{kind:22s} {len(shots)} shots in this build, first {shots[:2]}")
index_by_shot = index.set_index("shot")

In [ ]:
def raw_current(shot, column):
    """(time_ms, values) of one current column on its own time axis, in float64."""
    axis = shot[shot.machine.time_axis_column(column)]
    return np.asarray(axis, dtype=np.float64), np.asarray(shot[column], dtype=np.float64)


def plot_forms(ax, shot, column, frames, name, pad_ms=50.0):
    t, v = raw_current(shot, column)
    lo, hi = frames["time_ms"].min() - pad_ms, frames["time_ms"].max() + pad_ms
    inside = (t >= lo) & (t <= hi)
    ax.plot(t[inside], v[inside], lw=0.5, color="0.6", label="raw")
    ax.plot(frames["time_ms"], frames[f"{name}__nearest"], ".", ms=4, label="nearest")
    ax.plot(frames["time_ms"], frames[f"{name}__mean"], "x", ms=4, label="mean")
    ax.set_title(f"{column}")
    ax.set_xlabel("time (ms)")


fig, axes = plt.subplots(len(kinds), 2, figsize=(14, 3.2 * len(kinds)), squeeze=False)
for row, (kind, shots) in zip(axes, kinds.items()):
    shot_id = shots[0]
    info = index_by_shot.loc[shot_id]
    cfg = machine_config(info["machine"])
    columns = ["efit_times", "magnetics_time", "magnetics_plasma_current_times", cfg.ip_column, cfg.solenoid_column]
    shot = load_shot(info["path"], columns, align_ip=PARAMS["align_ip"])
    frames = read(info["split"], ["time_ms", "ip__nearest", "ip__mean", "sol__nearest", "sol__mean"], [shot_id])
    plot_forms(row[0], shot, cfg.ip_column, frames, "ip")
    plot_forms(row[1], shot, cfg.solenoid_column, frames, "sol")
    row[0].set_ylabel(f"{kind}\n{shot_id}\n{int(info['mag_rate_hz'])} Hz")
axes[0, 0].legend(loc="best")
fig.tight_layout()

In [ ]:
# T8: nearest minus mean relative to the spread of mean, per column and recording rate.
def t8_table(split, signals):
    cols = ["mag_rate_hz"] + [f"{s}__nearest" for s in signals] + [f"{s}__mean" for s in signals]
    df = read(split, cols)
    rows = []
    for rate, sub in df.groupby("mag_rate_hz"):
        for s in signals:
            diff = (sub[f"{s}__nearest"] - sub[f"{s}__mean"]).to_numpy(dtype=np.float64)
            spread = np.nanstd(sub[f"{s}__mean"].to_numpy(dtype=np.float64))
            rel = np.abs(diff[np.isfinite(diff)]) / spread if spread > 0 else np.array([])
            if rel.size:
                rows.append({"split": split, "rate_hz": int(rate), "signal": s, "frames": int(rel.size),
                             "p50": np.median(rel), "p90": np.percentile(rel, 90), "p99": np.percentile(rel, 99),
                             "max": rel.max()})
    return pd.DataFrame(rows)


t8 = []
for split in SPLITS:
    cfg = machine_config(MACHINE_OF[split])
    signals = ["ip", "sol", "tf", cfg.shaping_coil_columns[0], cfg.shaping_coil_columns[-1]]
    t8.append(t8_table(split, signals))
t8 = pd.concat(t8, ignore_index=True)
display(t8.round(5))

In [ ]:
# Distribution of the relative difference for Ip and the solenoid, per split and rate.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, signal in zip(axes, ["ip", "sol"]):
    for split in SPLITS:
        df = read(split, ["mag_rate_hz", f"{signal}__nearest", f"{signal}__mean"])
        for rate, sub in df.groupby("mag_rate_hz"):
            diff = (sub[f"{signal}__nearest"] - sub[f"{signal}__mean"]).to_numpy(dtype=np.float64)
            spread = np.nanstd(sub[f"{signal}__mean"].to_numpy(dtype=np.float64))
            rel = diff[np.isfinite(diff)] / spread if spread > 0 else np.array([])
            if rel.size:
                ax.hist(rel, bins=80, histtype="step", label=f"{split} {int(rate)} Hz ({rel.size})")
    ax.set_yscale("log")
    ax.set_title(f"({signal}__nearest - {signal}__mean) / std({signal}__mean)")
    ax.legend(fontsize=8)
fig.tight_layout()

## 3. Rates of change (A4)

The slope drawn against the raw trace through a current ramp: for one 20 kHz and one
2 kHz DIII-D shot, the frames with the largest absolute `ip__rate`, each with a segment
of slope `ip__rate` and length `rate_window_ms` centred on the frame. Then the spread of
`ip__rate` and of one shaping-coil rate on flat-top frames (`ip_over_peak` above 0.9),
compared between the 20 kHz and 2 kHz groups. A wider spread in one group means the
window is too short for that rate.

In [ ]:
def plot_slopes(ax, shot, frames, name, column, n_frames=6, pad_ms=60.0):
    t, v = raw_current(shot, column)
    top = frames.loc[frames[f"{name}__rate"].abs().nlargest(n_frames).index].sort_values("time_ms")
    lo, hi = top["time_ms"].min() - pad_ms, top["time_ms"].max() + pad_ms
    inside = (t >= lo) & (t <= hi)
    ax.plot(t[inside], v[inside], lw=0.6, color="0.6", label="raw")
    half = PARAMS["rate_window_ms"] / 2
    for _, f in top.iterrows():
        slope_per_ms = f[f"{name}__rate"] / 1000.0
        ax.plot([f["time_ms"] - half, f["time_ms"] + half],
                [f[f"{name}__mean"] - slope_per_ms * half, f[f"{name}__mean"] + slope_per_ms * half], "r-", lw=1.5)
        ax.plot(f["time_ms"], f[f"{name}__mean"], "k.", ms=5)
    ax.set_xlabel("time (ms)")
    ax.set_title(f"{column}: frames with the largest |{name}__rate|")


ramp_kinds = [k for k in ("d3d_20khz", "d3d_2khz", "mast_without_nan") if k in kinds]
fig, axes = plt.subplots(1, len(ramp_kinds), figsize=(6.5 * len(ramp_kinds), 4), squeeze=False)
for ax, kind in zip(axes[0], ramp_kinds):
    shot_id = kinds[kind][0]
    info = index_by_shot.loc[shot_id]
    cfg = machine_config(info["machine"])
    shot = load_shot(info["path"], ["efit_times", "magnetics_time", "magnetics_plasma_current_times", cfg.ip_column],
                     align_ip=PARAMS["align_ip"])
    frames = read(info["split"], ["time_ms", "ip__mean", "ip__rate"], [shot_id])
    plot_slopes(ax, shot, frames, "ip", cfg.ip_column)
    ax.set_ylabel(f"{kind} {shot_id}")
fig.tight_layout()

In [ ]:
# Spread of rates on flat-top frames, 20 kHz against 2 kHz (DIII-D training table).
FLAT_TOP = 0.9
QUANTILES = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
if "diii_d_train" in SPLITS:
    coil = machine_config("DIII-D").shaping_coil_columns[0]
    df = read("diii_d_train", ["mag_rate_hz", "ip_over_peak", "ip__rate", f"{coil}__rate"])
    flat = df[df["ip_over_peak"] > FLAT_TOP]
    print(f"{len(flat)} flat-top frames of {len(df)}")
    for col in ("ip__rate", f"{coil}__rate"):
        print(col)
        display(flat.groupby("mag_rate_hz")[col].quantile(QUANTILES).unstack().round(2))
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, col in zip(axes, ("ip__rate", f"{coil}__rate")):
        values = flat[col].dropna()
        lo, hi = values.quantile([0.005, 0.995])
        bins = np.linspace(lo, hi, 81)
        for rate, sub in flat.groupby("mag_rate_hz"):
            ax.hist(sub[col].dropna().clip(lo, hi), bins=bins, histtype="step", density=True,
                    label=f"{int(rate)} Hz, std {sub[col].std():.1f}")
        ax.set_title(f"{col} on flat-top frames")
        ax.set_yscale("log")
        ax.legend()
    fig.tight_layout()

## 4. Thomson in time (A3)

`__interp` against `__nearest` for the core electron pressure; coverage as a function of
the tolerance, per machine, from the stored time-distance columns; and the shots with
no usable Thomson. The validity flag in the table uses `thomson_tol_ms` of the build;
the coverage curve shows what any other tolerance would give without a rebuild.

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, len(SPLITS), figsize=(6 * len(SPLITS), 4.5), squeeze=False)
for ax, split in zip(axes[0], SPLITS):
    shots = sorted(index.loc[(index["split"] == split) & index["error"].isna(), "shot"])
    sample = sorted(rng.choice(shots, size=min(20, len(shots)), replace=False)) if shots else []
    df = read(split, ["ts_core_pe_mean__interp", "ts_core_pe_mean__nearest", "ts_core_dt_ms", "ts_core_gap_ms"], sample)
    sc = ax.scatter(df["ts_core_pe_mean__nearest"], df["ts_core_pe_mean__interp"], c=df["ts_core_dt_ms"], s=5,
                    cmap="viridis", vmin=0, vmax=PARAMS["thomson_tol_ms"])
    lim = np.nanpercentile(df[["ts_core_pe_mean__nearest", "ts_core_pe_mean__interp"]].to_numpy(), 99.5)
    ax.plot([0, lim], [0, lim], "k--", lw=0.8)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel("ts_core_pe_mean__nearest (kPa)")
    ax.set_ylabel("ts_core_pe_mean__interp (kPa)")
    ax.set_title(f"{split}, {len(sample)} shots")
    fig.colorbar(sc, ax=ax, label="ts_core_dt_ms")
fig.tight_layout()

In [ ]:
# Coverage against the tolerance: share of frames whose nearest valid slice is within tol, and
# share of shots with at least 90 percent of frames covered, per split and system.
TOLERANCES_MS = np.arange(0, 61, 2)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
coverage_rows = []
for split in SPLITS:
    df = read(split, ["shot", "ts_core_dt_ms", "ts_edge_dt_ms"])
    for system in ("core", "edge"):
        dt = df[f"ts_{system}_dt_ms"]
        frame_share = [float((dt <= tol).mean()) for tol in TOLERANCES_MS]
        shot_share = [float((df.assign(ok=dt <= tol).groupby("shot")["ok"].mean() >= 0.9).mean())
                      for tol in TOLERANCES_MS]
        style = "-" if system == "core" else "--"
        axes[0].plot(TOLERANCES_MS, frame_share, style, label=f"{split} {system}")
        axes[1].plot(TOLERANCES_MS, shot_share, style, label=f"{split} {system}")
        at_build = float((dt <= PARAMS["thomson_tol_ms"]).mean())
        coverage_rows.append({"split": split, "system": system, "frames_within_build_tol": at_build,
                              "frames_within_10ms": float((dt <= 10).mean()), "frames_without_any": float(dt.isna().mean())})
for ax, title in zip(axes, ("share of frames within tolerance", "share of shots with 90% of frames covered")):
    ax.axvline(PARAMS["thomson_tol_ms"], color="k", lw=0.8, ls=":")
    ax.set_xlabel("tolerance (ms)")
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.tight_layout()
display(pd.DataFrame(coverage_rows).round(4))

In [ ]:
no_thomson = index[index["error"].isna() & ~index["ts_core_any_valid"].fillna(False)]
print(f"{len(no_thomson)} shots with no valid core profile")
display(no_thomson[["split", "shot", "n_frames", "ts_edge_any_valid"]])
no_edge = index[index["error"].isna() & ~index["ts_edge_any_valid"].fillna(False)]
print(f"{len(no_edge)} shots with no valid edge profile")

## 5. Column profile (B1)

For every column: range, percentiles, missing fraction and share of exact zeros, from
`column_profile.parquet`. Then histograms of the shared quantities for DIII-D training,
DIII-D test and MAST side by side, and the shaping-coil currents per machine. The
question for each input column is whether it is usable as it is, needs scaling, or is
dropped.

In [ ]:
profile_path = REPORT_DIR / "column_profile.parquet"
if profile_path.is_file():
    profile = pd.read_parquet(profile_path)
    show = ["split", "column", "role", "n", "missing_frac", "zero_frac", "min", "p01", "p50", "p99", "max"]
    for split in SPLITS:
        sub = profile[(profile["split"] == split) & profile["role"].isin(["input", "flag", "label"])]
        print(split)
        display(sub[show].round(4).reset_index(drop=True))
else:
    print(f"no profile at {profile_path}; run scripts/profile_frame_table.py --table-dir {TABLE_DIR} --out {REPORT_DIR}")

In [ ]:
SHARED = ["ip__mean", "sol__mean", "tf__mean", "ip__rate", "sol__rate", "tf__rate",
          "ts_core_te_peak__interp", "ts_core_ne_peak__interp", "ts_core_pe_mean__interp",
          "ts_edge_te_mean__interp", "ts_edge_ne_mean__interp", "dt_prev_ms", "n_frames"]
shared = {split: read(split, SHARED) for split in SPLITS}
ncol = 3
nrow = int(np.ceil(len(SHARED) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 3.6 * nrow))
for ax, col in zip(axes.ravel(), SHARED):
    values = pd.concat([shared[s][col] for s in SPLITS]).dropna()
    lo, hi = values.quantile([0.001, 0.999]) if len(values) else (0, 1)
    if lo == hi:
        lo, hi = lo - 1, hi + 1
    bins = np.linspace(lo, hi, 81)
    for split in SPLITS:
        v = shared[split][col].dropna()
        ax.hist(v.clip(lo, hi), bins=bins, histtype="step", density=True, label=f"{split} ({v.notna().sum()})")
    ax.set_title(col)
    ax.set_yscale("log")
    ax.legend(fontsize=7)
for ax in axes.ravel()[len(SHARED):]:
    ax.axis("off")
fig.tight_layout()

In [ ]:
# Shaping-coil currents per machine: distribution of the windowed mean per coil.
fig, axes = plt.subplots(1, len(SPLITS), figsize=(6.5 * len(SPLITS), 4.5), squeeze=False)
for ax, split in zip(axes[0], SPLITS):
    cfg = machine_config(MACHINE_OF[split])
    cols = [f"{c}__mean" for c in cfg.shaping_coil_columns] + [f"{c}__mean" for c in cfg.extra_current_columns]
    df = read(split, cols)
    data = [df[c].dropna().to_numpy() for c in cols]
    ax.boxplot(data, whis=(1, 99), showfliers=False)
    ax.set_xticks(range(1, len(cols) + 1))
    ax.set_xticklabels([c.replace("magnetics_", "").replace("__mean", "").replace("_current", "") for c in cols],
                       rotation=90, fontsize=8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(f"{split}: shaping coils, kA-turn (1 to 99 percent whiskers)")
fig.tight_layout()

## 6. Relationships (B6), DIII-D training frames

A first read on which targets follow simple combinations of the inputs: flux range
(`psi_max` minus `psi_min`) against absolute Ip; `efit_beta_n` against the core electron
pressure; `efit_q95` against absolute `tf` over absolute `ip`; and the correlation matrix
of the coil currents. A fixed-seed sample of shots keeps the scatter plots readable.

In [ ]:
if "diii_d_train" in SPLITS:
    cfg = machine_config("DIII-D")
    rng = np.random.default_rng(SEED)
    train_shots = sorted(index.loc[(index["split"] == "diii_d_train") & index["error"].isna(), "shot"])
    sample = sorted(rng.choice(train_shots, size=min(300, len(train_shots)), replace=False))
    rel = read("diii_d_train", ["shot", "ip__mean", "tf__mean", "ip_over_peak", "psi_min", "psi_max",
                                "efit_beta_n", "efit_q95", "ts_core_pe_mean__interp", "ts_core_valid"], sample)
    rel = rel[rel["ip_over_peak"] > 0.5]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].scatter(rel["ip__mean"].abs(), rel["psi_max"] - rel["psi_min"], s=2, alpha=0.3)
    axes[0].set_xlabel("|ip__mean| (kA)")
    axes[0].set_ylabel("psi_max - psi_min (Wb/rad)")
    ok = rel["ts_core_valid"]
    axes[1].scatter(rel.loc[ok, "ts_core_pe_mean__interp"], rel.loc[ok, "efit_beta_n"], s=2, alpha=0.3)
    axes[1].set_xlabel("ts_core_pe_mean__interp (kPa)")
    axes[1].set_ylabel("efit_beta_n")
    axes[1].set_ylim(np.nanpercentile(rel["efit_beta_n"], [0.5, 99.5]))
    ratio = rel["tf__mean"].abs() / rel["ip__mean"].abs()
    axes[2].scatter(ratio, rel["efit_q95"], s=2, alpha=0.3)
    axes[2].set_xlabel("|tf__mean| / |ip__mean|")
    axes[2].set_ylabel("efit_q95")
    axes[2].set_ylim(0, np.nanpercentile(rel["efit_q95"], 99.5))
    axes[2].set_xlim(0, np.nanpercentile(ratio, 99.5))
    fig.suptitle(f"{len(sample)} training shots, frames with ip_over_peak above 0.5")
    fig.tight_layout()
    for x, y, name in ((rel["ip__mean"].abs(), rel["psi_max"] - rel["psi_min"], "flux range vs |Ip|"),
                       (rel.loc[ok, "ts_core_pe_mean__interp"], rel.loc[ok, "efit_beta_n"], "betaN vs core pe"),
                       (ratio, rel["efit_q95"], "q95 vs |tf|/|ip|")):
        m = np.isfinite(x) & np.isfinite(y)
        print(f"{name:22s} Pearson r = {np.corrcoef(x[m], y[m])[0, 1]:.3f} over {int(m.sum())} frames")

In [ ]:
if "diii_d_train" in SPLITS:
    cfg = machine_config("DIII-D")
    coil_cols = ["ip__mean", "sol__mean", "tf__mean"] + [f"{c}__mean" for c in cfg.shaping_coil_columns]
    coils = read("diii_d_train", coil_cols + ["ip_over_peak"])
    coils = coils[coils["ip_over_peak"] > 0.5].drop(columns="ip_over_peak")
    corr = coils.corr()
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    labels = [c.replace("magnetics_", "").replace("__mean", "") for c in corr.columns]
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    fig.colorbar(im, ax=ax, label="Pearson r")
    ax.set_title(f"Coil current correlations, {len(coils)} DIII-D training frames with ip_over_peak above 0.5")
    fig.tight_layout()
    pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().dropna()
    print("most correlated pairs:")
    display(pairs.abs().sort_values(ascending=False).head(12).round(3))

## 7. Findings

The decision reached on each item of the phase, with the evidence from the sections
above, and anything unexpected. Filled in after the full run; new findings are added to
`docs/findings.md`.

| Decision | Options | Decision | Evidence |
| --- | --- | --- | --- |
| Resampling of currents (A2) | nearest sample; windowed mean, and its window length | | sections 2 and 3; ridge score in the baseline phase |
| Rate-of-change window (A4) | 10 ms, or longer if the two recording rates differ in noise | | section 3 |
| Thomson in time (A3) | interpolation between valid slices; nearest slice within a tolerance | | section 4 |
| Thomson validity tolerance | 20 ms, or another value read off the coverage curve | | section 4 |
| Usable inputs (B1) | per column: usable as is, needs scaling, or dropped | | section 5 |
| First read on linearity (B6) | which targets follow simple combinations of the inputs | | section 6 |

Unexpected observations:

-